Step 4: Random Forest

Regression

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import shap

Read data

In [16]:
file = 'hourly_columbia_weather.parquet'

# Read parquet into dataframe
df = pd.read_parquet(file)

print(df.head())

                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-01-02 00:00:00  10.0   1.1  54.0   0.0   20.0  14.8  1019.2   3.0   
2024-01-02 01:00:00   8.9  -2.1  46.0   0.0  350.0  13.0  1020.2   3.0   
2024-01-02 02:00:00   6.7  -3.8  47.0   0.0    2.0   7.0  1021.2   3.0   
2024-01-02 03:00:00   6.1  -3.8  49.0   0.0  350.0   7.6  1022.2   2.0   
2024-01-02 04:00:00   5.6  -4.3  49.0   0.0  350.0   7.6  1022.8   3.0   

                     prcp_flag       day_sin  ...  wspd_roll12  prcp_roll3  \
2024-01-02 00:00:00          0 -9.634256e-12  ...     9.116667         0.0   
2024-01-02 01:00:00          0  2.588190e-01  ...     9.900000         0.0   
2024-01-02 02:00:00          0  5.000000e-01  ...    10.533333         0.0   
2024-01-02 03:00:00          0  7.071068e-01  ...    11.116667         0.0   
2024-01-02 04:00:00          0  8.660254e-01  ...    10.816667         0.0   

                     prcp_roll6  prcp_roll12  dwpt_roll3  dwpt_roll6  \
2024-01-02 00:

Prepare data, split into x and y

In [17]:
# defining target (Y) and explanatory variables (X)
exclude_reg = ['wdir', 'coco', 'prcp_flag']
exclude_time = exclude_reg + ['day_sin', 'day_cos', 'week_sin', 'week_cos', 'year_sin',
                               'year_cos']
exclude_lag = exclude_reg + [ 'rhum_lag1', 'rhum_lag3', 'rhum_lag6',
       'rhum_lag24', 'pres_lag1', 'pres_lag3', 'pres_lag6', 'pres_lag24',
       'wspd_lag1', 'wspd_lag3', 'wspd_lag6', 'dwpt_lag1', 'dwpt_lag3', 'dwpt_lag6', 'wdir_sin_lag1',
       'wdir_sin_lag3', 'wdir_sin_lag6', 'wdir_cos_lag1', 'wdir_cos_lag3',
       'wdir_cos_lag6']
exclude_roll = exclude_reg + [
       'rhum_roll3', 'rhum_roll6', 'rhum_roll12', 'wspd_roll3', 'wspd_roll6',
       'wspd_roll12', 'prcp_roll3', 'prcp_roll6', 'prcp_roll12', 'dwpt_roll3',
       'dwpt_roll6', 'dwpt_roll12', 'prcp_sum3', 'prcp_sum6', 'prcp_sum12']
exclude_temp = exclude_reg + ['temp_roll3', 'temp_roll6', 'temp_roll12', 'temp_lag1', 'temp_lag3',
       'temp_lag6', 'temp_lag24']

target_name = ['temp']
###
# edit exclusion
###
exclude = target_name +  exclude_temp + exclude_time + exclude_roll + exclude_lag 
X = df.drop(columns=exclude)
Y = df[target_name]

print(X)
print(Y)

                     dwpt  rhum  prcp  wspd    pres  wdir_sin  wdir_cos  \
2024-01-02 00:00:00   1.1  54.0   0.0  14.8  1019.2   0.34202  0.939693   
2024-01-02 01:00:00  -2.1  46.0   0.0  13.0  1020.2 -0.173648  0.984808   
2024-01-02 02:00:00  -3.8  47.0   0.0   7.0  1021.2  0.034899  0.999391   
2024-01-02 03:00:00  -3.8  49.0   0.0   7.6  1022.2 -0.173648  0.984808   
2024-01-02 04:00:00  -4.3  49.0   0.0   7.6  1022.8 -0.173648  0.984808   
...                   ...   ...   ...   ...     ...       ...       ...   
2024-12-31 20:00:00  12.8  57.0   0.0  20.5  1004.7 -0.866025      -0.5   
2024-12-31 21:00:00  11.6  51.0   0.0  16.6  1004.5 -0.766044 -0.642788   
2024-12-31 22:00:00  10.0  49.0   0.0  20.5  1004.9 -0.766044 -0.642788   
2024-12-31 23:00:00  10.7  59.0   0.0  11.2  1005.2      -0.5 -0.866025   
2025-01-01 00:00:00  11.3  65.0   0.0  15.0  1006.0      -0.5 -0.866025   

                     prcp_lag1  prcp_lag3  prcp_lag6  
2024-01-02 00:00:00        0.0        0.0   

Train/test split

In [18]:
# train/test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42)

# fitting a RF model
model = RandomForestRegressor(random_state=0)
model.fit(X_train, Y_train)

c:\Users\Student Account\anaconda3\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Get predictions

In [19]:
# getting forecasts for the test set
y_pred = model.predict(X_test)

# computing metrics
r2 = r2_score(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
mse = mean_squared_error(Y_test, y_pred)
rmse = np.sqrt(mse)

comparison = np.column_stack((Y_test, y_pred))
print('actual vs pred')
print(comparison)

actual vs pred
[[ 8.9    8.92 ]
 [14.4   14.424]
 [22.2   21.204]
 ...
 [10.    10.041]
 [23.9   23.9  ]
 [ 4.4    4.388]]


In [20]:
print(f'R2: \t{r2:.4f}')
print(f'MAE: \t{mae:.4f}')
print(f'MSE:  \t{mse:.4f}')
print(f'RMSE: \t{rmse:.4f}')

R2: 	0.9995
MAE: 	0.0963
MSE:  	0.0371
RMSE: 	0.1927


In [21]:
df[['prcp', 'prcp_roll3', 'prcp_roll6', 'prcp_roll12']].corr()

,prcp,prcp_roll3,prcp_roll6,prcp_roll12
prcp,1.000000,0.554017,0.486632,0.400589
prcp_roll3,0.554017,1.000000,0.875557,0.694916
prcp_roll6,0.486632,0.875557,1.000000,0.853412
prcp_roll12,0.400589,0.694916,0.853412,1.000000


SHAP

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)



In [ ]:
shap.summary_plot(shap_values, X_test) 